In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, StrMethodFormatter
from IPython.display import display

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "data" / "processed" / "btc_spy_daily.csv").exists())
RESULTS_DIR = project_root / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TOTAL_BUDGET = 12_000.0
ROLLING_HORIZONS_YEARS = [1, 3, 5]
assert TOTAL_BUDGET > 0


In [ ]:
prices = pd.read_csv(project_root / "data" / "processed" / "btc_spy_daily.csv",
                     index_col="date", parse_dates=True).rename(
    columns={"price_btc": "BTC", "price_spy": "SPY"})[["BTC", "SPY"]]
assert prices.index.is_monotonic_increasing and prices.index.is_unique
assert np.isfinite(prices.to_numpy()).all() and prices.gt(0).all().all()
assert len(prices) > 2
monthly_dates = prices.groupby(prices.index.to_period("M")).head(1).index
period_label = f"{prices.index[0]:%Y-%m-%d} to {prices.index[-1]:%Y-%m-%d}"
print(period_label)
print(f"Budget per strategy: ${TOTAL_BUDGET:,.2f}")
print(f"{len(monthly_dates)} purchases of ${TOTAL_BUDGET / len(monthly_dates):,.6f} each")
print(f"First purchase: {monthly_dates[0].date()}; last purchase: {monthly_dates[-1].date()}")


In [ ]:
def simulate_deployment(price, purchase_dates, budget):
    """Compare buy-and-hold with equal installments; idle cash earns zero."""
    purchase_dates = pd.DatetimeIndex(purchase_dates)
    if len(purchase_dates) == 0 or not purchase_dates.is_unique:
        raise ValueError("Purchase dates must be nonempty and unique.")
    if not purchase_dates.isin(price.index).all():
        raise ValueError("Every purchase date must have an observed price.")
    deposits = pd.Series(0.0, index=price.index)
    deposits.loc[purchase_dates] = budget / len(purchase_dates)
    # Correct only floating-point summation residue at the final purchase.
    deposits.loc[purchase_dates[-1]] += budget - deposits.sum()
    deployed = deposits.cumsum()
    units = (deposits / price).cumsum()
    cash = budget - deployed
    invested = units * price
    history = pd.DataFrame({
        "Lump sum wealth": budget * price / price.iloc[0],
        "Monthly invested value": invested,
        "Monthly cash": cash,
        "Monthly total wealth": invested + cash,
        "Cumulative contributions": deployed,
        "Contribution": deposits,
    })
    return history

histories = {asset: simulate_deployment(prices[asset], monthly_dates, TOTAL_BUDGET)
             for asset in prices}
display(histories["BTC"].head().style.format("${:,.2f}"))


In [ ]:
def contribution_xirr(contributions, terminal_value, terminal_date):
    contributions = contributions[contributions > 0]
    elapsed = np.asarray((pd.Timestamp(terminal_date) - contributions.index).days / 365.25)
    amounts = contributions.to_numpy(dtype=float)
    if len(amounts) == 0 or np.any(elapsed < 0) or not np.any(elapsed > 0):
        raise ValueError("XIRR requires contributions before the terminal date.")
    if terminal_value <= amounts[elapsed == 0].sum():
        raise ValueError("No finite XIRR above -100% for these cash flows.")
    def residual(log_growth):
        terms = np.log(amounts) + elapsed * log_growth
        peak = terms.max()
        return peak + np.log(np.exp(terms - peak).sum()) - np.log(terminal_value)
    low, high = -1.0, 1.0
    for _ in range(60):
        if residual(low) <= 0 <= residual(high):
            break
        low *= 2
        high *= 2
    else:
        raise ValueError("Could not bracket the XIRR root.")
    for _ in range(200):
        mid = (low + high) / 2
        if residual(mid) > 0:
            high = mid
        else:
            low = mid
    return np.expm1((low + high) / 2)


In [ ]:
years = (prices.index[-1] - prices.index[0]).days / 365.25
rows = []
for asset, history in histories.items():
    for strategy, wealth_col in [("Lump sum", "Lump sum wealth"),
                                ("Monthly", "Monthly total wealth")]:
        wealth = history[wealth_col]
        contributions = history["Contribution"] if strategy == "Monthly" else pd.Series(
            [TOTAL_BUDGET], index=prices.index[:1])
        ending_invested = (history["Monthly invested value"].iloc[-1]
                           if strategy == "Monthly" else wealth.iloc[-1])
        rows.append({
            "Asset": asset, "Strategy": strategy,
            "Budget": TOTAL_BUDGET,
            "Final wealth": wealth.iloc[-1],
            "Profit": wealth.iloc[-1] - TOTAL_BUDGET,
            "Total return": wealth.iloc[-1] / TOTAL_BUDGET - 1,
            "Whole-account CAGR": (wealth.iloc[-1] / TOTAL_BUDGET) ** (1 / years) - 1,
            "Invested-sleeve XIRR": contribution_xirr(contributions, ending_invested, prices.index[-1]),
            "Whole-account max drawdown": (wealth / wealth.cummax() - 1).min(),
        })
strategy_summary = pd.DataFrame(rows).set_index(["Asset", "Strategy"])
summary_formats = {col: "${:,.2f}" for col in ["Budget", "Final wealth", "Profit"]}
summary_formats.update({col: "{:.2%}" for col in ["Total return", "Whole-account CAGR",
    "Invested-sleeve XIRR", "Whole-account max drawdown"]})
display(strategy_summary.style.format(summary_formats).set_caption(period_label))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (asset, history) in zip(axes, histories.items()):
    history[["Lump sum wealth", "Monthly total wealth"]].plot(ax=ax)
    ax.set_title(asset)
    ax.set_xlabel("Date")
    ax.set_ylabel("Total wealth (USD)")
    ax.yaxis.set_major_formatter(StrMethodFormatter("${x:,.0f}"))
    ax.grid(alpha=0.25)
fig.suptitle(f"Equal ${TOTAL_BUDGET:,.0f} upfront budgets; idle cash earns 0%")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "contribution_wealth.png", dpi=300)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (asset, history) in zip(axes, histories.items()):
    wealth = history[["Lump sum wealth", "Monthly total wealth"]]
    (wealth / wealth.cummax() - 1).plot(ax=ax)
    ax.set_title(asset)
    ax.set_xlabel("Date")
    ax.set_ylabel("Whole-account drawdown")
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "contribution_drawdowns.png", dpi=300)
plt.show()


In [ ]:
cohort_rows = []
for horizon in ROLLING_HORIZONS_YEARS:
    for start in monthly_dates:
        anniversary = start + pd.DateOffset(years=horizon)
        if anniversary > prices.index[-1]:
            continue
        end = prices.index[prices.index.searchsorted(anniversary)]
        purchase_dates = monthly_dates[monthly_dates >= start][:12 * horizon]
        if len(purchase_dates) != 12 * horizon or purchase_dates[-1] >= end:
            raise ValueError("Unexpected purchase schedule; inspect missing calendar months.")
        for asset in prices:
            segment = prices.loc[start:end, asset]
            lump_final = TOTAL_BUDGET * segment.iloc[-1] / segment.iloc[0]
            monthly_final = (TOTAL_BUDGET / len(purchase_dates)
                             / segment.loc[purchase_dates]).sum() * segment.iloc[-1]
            relative_gap = monthly_final / lump_final - 1
            cohort_rows.append({
                "Asset": asset, "Horizon years": horizon, "Start": start, "End": end,
                "Lump sum final": lump_final, "Monthly final": monthly_final,
                "Monthly minus lump dollars": monthly_final - lump_final,
                "Monthly relative wealth advantage": relative_gap,
                "Winner": "Tie" if np.isclose(relative_gap, 0, atol=1e-10, rtol=0)
                          else ("Monthly" if relative_gap > 0 else "Lump sum"),
            })
cohorts = pd.DataFrame(cohort_rows)
if cohorts.empty:
    raise ValueError("No complete historical windows for the requested horizons.")

groups = cohorts.groupby(["Asset", "Horizon years"])
cohort_summary = groups.agg(
    Scenarios=("Winner", "size"),
    First_start=("Start", "min"), Last_start=("Start", "max"),
    Monthly_win_rate=("Winner", lambda x: x.eq("Monthly").mean()),
    Lump_sum_win_rate=("Winner", lambda x: x.eq("Lump sum").mean()),
    Tie_rate=("Winner", lambda x: x.eq("Tie").mean()),
    Median_monthly_advantage=("Monthly relative wealth advantage", "median"),
    Worst_monthly_advantage=("Monthly relative wealth advantage", "min"),
    Best_monthly_advantage=("Monthly relative wealth advantage", "max"),
)
cohort_formats = {col: "{:.2%}" for col in cohort_summary if "rate" in col or "advantage" in col}
cohort_formats.update({"First_start": "{:%Y-%m-%d}", "Last_start": "{:%Y-%m-%d}"})
display(cohort_summary.style.format(cohort_formats))


In [ ]:
fig, axes = plt.subplots(len(ROLLING_HORIZONS_YEARS), 1,
                         figsize=(12, 3.5 * len(ROLLING_HORIZONS_YEARS)), squeeze=False)
for ax, horizon in zip(axes[:, 0], ROLLING_HORIZONS_YEARS):
    selected = cohorts[cohorts["Horizon years"] == horizon]
    for asset in prices:
        data = selected[selected["Asset"] == asset]
        ax.plot(data["Start"], data["Monthly relative wealth advantage"], label=asset)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"{horizon}-year horizon: above zero favors monthly deployment")
    ax.set_xlabel("Scenario starting date")
    ax.set_ylabel("Monthly / lump wealth − 1")
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.legend()
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "contribution_rolling_comparison.png", dpi=300)
plt.show()


In [ ]:
strategy_summary.to_csv(RESULTS_DIR / "contribution_strategy_summary.csv")
pd.concat(histories, names=["Asset", "Date"]).to_csv(RESULTS_DIR / "contribution_history.csv")
cohorts.to_csv(RESULTS_DIR / "contribution_rolling_scenarios.csv", index=False)
cohort_summary.to_csv(RESULTS_DIR / "contribution_rolling_summary.csv")
assumptions = {"budget_per_strategy": TOTAL_BUDGET, "price_period": period_label,
    "full_period_purchases": len(monthly_dates), "rolling_horizons_years": ROLLING_HORIZONS_YEARS,
    "purchase_timing": "first available shared date of each month at recorded closing price",
    "idle_cash_yield": 0, "fees_taxes_spreads": 0, "fractional_units": True,
    "spy": "adjusted-price dividend-reinvestment proxy; units are synthetic",
    "xirr": "invested sleeve only; whole-account CAGR includes idle cash",
    "rolling_endpoint": "first shared date on or after calendar anniversary"}
(RESULTS_DIR / "contribution_assumptions.json").write_text(json.dumps(assumptions, indent=2))
report = (
    '<!doctype html><html><head><meta charset="utf-8"><title>Contribution strategies</title>'
    '<style>body{font-family:system-ui;max-width:1400px;margin:35px auto;padding:20px;}'
    'table{border-collapse:collapse;font-size:13px;}th,td{padding:8px;border-bottom:1px solid #ddd;}'
    'img{max-width:100%;}</style></head><body><h1>BTC and SPY: lump sum vs monthly deployment</h1>'
    f'<p>{period_label}. Equal upfront budgets of ${TOTAL_BUDGET:,.2f}. '
    f'{len(monthly_dates)} equal monthly purchases in the full-period scenario.</p>'
    '<p>Idle cash earns 0%; no fees, taxes or spreads; fractional units permitted. '
    'SPY adjusted prices proxy reinvested dividends. This comparison assumes all capital is available initially.</p>'
    '<h2>Full-period results</h2>' + strategy_summary.style.format(summary_formats).to_html()
    + '<p>Whole-account wealth includes idle cash. XIRR covers only the invested sleeve and therefore '
    'does not measure the opportunity cost of holding cash. Drawdowns use whole-account wealth.</p>'
    '<img src="contribution_wealth.png" alt="Strategy wealth">'
    '<img src="contribution_drawdowns.png" alt="Whole-account drawdowns">'
    '<h2>Rolling entry-date analysis</h2>' + cohort_summary.style.format(cohort_formats).to_html()
    + '<p>Monthly advantage = monthly ending wealth / lump-sum ending wealth minus 1. '
    'Overlapping windows are not independent. Horizons have different available start dates and deployment durations. '
    'Observed win rates are not future probabilities. Endpoints use the first shared date on or after the anniversary.</p>'
    '<img src="contribution_rolling_comparison.png" alt="Results across starting dates">'
    '<p>Zero cash interest can disadvantage gradual deployment. Shared closes miss intraday/weekend extremes. '
    'Historical results are sensitive to the sample and are not forecasts.</p></body></html>'
)
(RESULTS_DIR / "contribution_report.html").write_text(report, encoding="utf-8")
print(f"Saved four CSVs, assumptions JSON, three charts and an HTML report to {RESULTS_DIR}")
